# KI-FSPrompt Online-Inference

Trying KI-FSPrompt for single Text input. This notebook demonstrates how
ki-fsprompt operates. While the offline-inference version of KI-FSPrompt
proceeds through the stages, by processing a complete batch of texts in
every stage with vllm-Offline-Inference, loading LLMs into GPU-memory on demand,
this demo demonstrates the steps for a single input-text, using online-inference,
which requires that a LLM is served permanently on some API-Url. 

## Prerequisites

You need to start all three services used by ki-fsprompt, as specified in the
`docker-compose.yaml`.

1. Weaviate to serve the vector storage for `retrieve` and `map`
2. Huggingface-TEI to serve the embedding generation for `retrieve` and `map`
3. vLLM to serve a generative LLM for `complete` and `rank`

Also we assume that the weaviate collections for `retrieve` and `map` have been 
created using the respective dvc-stages in the `train` pipeline. If these do 
not exist, you can create them by running:

```bash
dvc repro -sf train/dvc.yaml:create_vocab_collection \
  train/dvc.yaml:create_train_collection
```

In [ ]:
# Only needed for debugging with VS Code
import debugpy
debugpy.listen(5678)
debugpy.wait_for_client() 

## Retrieval Stage

Im ersten Schritt wird die Vektordatenbank abgefragt, die die Texte aus dem
Train Datensatz, inklusive ihrer Schlagworte beinhaltet.

In [2]:
import pandas as pd
from src.retrieve import ExampleRetriever

input_text_data = {
    "text": ["Untersuchungen zum Einsatz der NEFA und der BHB zur Stoffwechselüberwachung von Transitkühen unter besonderer Berücksichtigung von gepoolten Serumproben"],
    "doc_id": ['1'],
    "label_ids": [['1']],
    "label_texts": [["Energiestoffwechsel", "Stichprobenprüfung", "Blutserum", "Milchkuh", "Mutterkuh", "Tiergesundheit"]]
}

retriever = ExampleRetriever(
        input_text_data = pd.DataFrame(input_text_data),
        output_file=None,
        n_examples=5,
        collection_name="title_train",
        host="8090",
        debug=False,
    )

retrieved_texts = retriever.retrieve_examples()
retrieved_texts

0it [00:00, ?it/s]

1it [00:00, 34.36it/s]


,doc_id,text,label_ids,label_texts,prompt_doc_id,prompt_text,prompt_labels,prompt_label_texts,similarity
0,1,Untersuchungen zum Einsatz der NEFA und der BH...,[1],"[Energiestoffwechsel, Stichprobenprüfung, Blut...",990221601,Nutzung transgener Tiermodelle mit Transportde...,"040385752, 040395618, 041406605, 042034337, 04...",Membrantransport; Missbildung; Tiermodell; Arz...,0.389248
1,1,Untersuchungen zum Einsatz der NEFA und der BH...,[1],"[Energiestoffwechsel, Stichprobenprüfung, Blut...",1084634597,Untersuchungen zur Schwefelversorgung von Milc...,"041699173, 041700406, 041803809, 947676724",Milchkuhhaltung; Mineralstoffversorgung; Schwe...,0.397323
2,1,Untersuchungen zum Einsatz der NEFA und der BH...,[1],"[Energiestoffwechsel, Stichprobenprüfung, Blut...",117058280X,Untersuchungen zu Einflüssen auf die Beta-Hyd...,"040392643, 041354117, 04141246X, 04146088X, 04...",Milch; Infrarotspektroskopie; Acetonämie; Blut...,0.402472
3,1,Untersuchungen zum Einsatz der NEFA und der BH...,[1],"[Energiestoffwechsel, Stichprobenprüfung, Blut...",1169832172,Untersuchungen zur Versorgungslage von Milchku...,"040015734, 041403002, 041699165, 041837622, 95...",Aluminium; Barium; Milchkuh; Strontium; Spuren...,0.406996
4,1,Untersuchungen zum Einsatz der NEFA und der BH...,[1],"[Energiestoffwechsel, Stichprobenprüfung, Blut...",1068209348,Untersuchungen zur Beurteilung der Molybdänve...,"041699165, 04170407X, 951478877",Milchkuh; Molybdän; Spurenelementversorgung,0.413001


## Erzeuge einen Prompt aus dem neuen Text und den Beispielen

Wir benutzen die PromptBuilder Funktion, um aus den extrahierten Beispielen
`messages` für eine LLM-Anfrage zu konstruieren

In [3]:
from src.PromptBuilder import IndividualPromptBuilder

prompt_builder = IndividualPromptBuilder(
    parsed_prompt_file=retrieved_texts,
    custom_instruction="Bitte extrahiere die Schlüsselwörter aus dem Text, wie in den Beispielen. Antworte im selben JSON-Format",
    debug=False,
)

messages = prompt_builder.build_messages(doc_id=str(input_text_data["doc_id"][0]), text=input_text_data["text"][0])
messages


[{'role': 'system',
  'content': 'Bitte extrahiere die Schlüsselwörter aus dem Text, wie in den Beispielen. Antworte im selben JSON-Format'},
 {'role': 'user',
  'content': 'Nutzung transgener Tiermodelle mit Transportdefekten zur Analyse der hepatobiliären Elimination und Organverteilung von Arzneistoffen und Toxinen'},
 {'role': 'assistant',
  'content': '{"keywords": ["Membrantransport", "Missbildung", "Tiermodell", "Arzneimittelverteilung", "Transgene Tiere"]}'},
 {'role': 'user',
  'content': 'Untersuchungen zur Schwefelversorgung von Milchkühen. Evaluation of the sulfur status in dairy cows'},
 {'role': 'assistant',
  'content': '{"keywords": ["Milchkuhhaltung", "Mineralstoffversorgung", "Schwefel", "Tiergesundheit"]}'},
 {'role': 'user',
  'content': 'Untersuchungen zu Einflüssen auf die Beta-Hydroxybutyrat-Konzentration im Blut und die Infrarotspektren der Milch sowie auf die Aussagekraft eines Frühwarnsystems für das Auftreten von Ketonämien bei Milchkühen'},
 {'role': 

## Führe den Completion-Schritt durch

Hier weichen wir vom Quellcode von KI-FSPrompt ab, weil das Skript `src/completion.py`
mit Batch-Processing und Offline-Inferenz arbeitet.

In [ ]:
LLM_API_URL = "http://localhost:9513/v1"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

from openai import OpenAI

client = OpenAI(
  base_url=LLM_API_URL, 
  api_key = "must be a non-empty string, but is not required for vLLM")

keyword_schema = {
    "type": "object",
    "properties": {
        "keywords": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 20,
        }
    },
    "required": ["keywords"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model = MODEL_NAME,
    messages=messages,
    extra_body={
      "guided_json": keyword_schema,
      # chat_template not supported by mistral-7B-0p3
      #"chat_template_kwargs": {"enable_thinking": False}
      }
)

keywords = response.choices[0].message.content
print(keywords)

 {"keywords": ["NEFA", "BHB", "Transitkuh", "Stoffwechsel", "Serumproben"]}


## Mapping auf GND Normdaten

Wir verwenden nun die weaviate-Collection, welche die GND mit Vektorisierung
zur Verfügung stellt, um die freien Schlagwörter auf mögliche GND-Einträge zu mappen.

In [6]:
from src.mapping import Mapping
import json
import weaviate

db_connection = weaviate.connect_to_local(port=8087)

hyperparameters = {
  "host": '8090',
  "alpha": 0.7,
  "use_phrase": False,
  "search": "hybrid"
}

mapping = Mapping(
    hyperparameters=hyperparameters,
    collection_name="ki_fsprompt_vocab",
    phrase=None,
    debug=False,
    db_connection=db_connection,
)

keyword_list = json.loads(keywords)["keywords"]

results = []
for keyword in keyword_list:
    item = mapping.query_vector_database(candidate=keyword)
    for label, info in item.items():
        results.append({"keyword": keyword, **info})


mapped_keywords = pd.DataFrame(results)

db_connection.close()
mapped_keywords

,keyword,label_id,term,hybrid_score,cosine_similarity
0,NEFA,000380989,REFA,0.7,0.755174
1,BHB,955632706,BHB,1.0,0.999983
2,Transitkuh,042735610,Transitzeit,0.7,0.709528
3,Stoffwechselüberwachung,04057699X,Stoffwechsel,0.7,0.784008
4,Serumproben,041810627,Serumproteine,0.7,0.776001
